# E6 / E7 / E8 — Phase 2 Baselines

**Experiment IDs:** `E6` (gradient boosting), `E7` (sequence model), `E8` (steelmanned MC-dropout).
**Specification:** `EXPERIMENT_PLAN.md` §E6–E8; artifact named in `IMPLEMENTATION_PLAYBOOK.md`
(`reports/02_baselines.html`). **Governing rules:** `CLAUDE.md`.

**What Phase 2 is for.** These are *baseline reproductions*, not contributions. `PROJECT_KNOWLEDGE.md`
§6 is explicit that this project claims no point-prediction novelty — the predictors exist to sit
underneath the Phase-3 conformal layer, and E8 exists so that its **coverage** can be audited. A
weak point score is not a failure of this phase; an unaudited uncertainty claim would be.

**Test-set discipline.** The official test set is read **exactly once per experiment**, at the end.
Every hyperparameter, target parameterisation and decision threshold is selected on `val_inner`,
which is carved out of the `fit` pool so Phase 3's `calibration` / `self_test` subsets stay
untouched (`data.training_pool_splits`).

**Gate 1 context.** Group-conditional analysis is out of scope (E13 dropped, Gate 1 PIVOT).
`mission_id` is used only as an ordinary feature. Nominal levels are {80%, 90%, 95%} with 90%
primary (Q-STAT-01, resolved).

In [ ]:
# --- Setup, configuration, and provenance stamp (invariant I4) -------------------------------
import json, subprocess, sys, warnings
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal import baselines as bl
from kelvins_conformal import data as kcdata
from kelvins_conformal import features as feat
from kelvins_conformal.models import experiments as EX
from kelvins_conformal.models import runner as R

cfg = load_config()
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha() -> str:
    try:
        out = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT),
                             capture_output=True, text=True, check=True)
        return out.stdout.strip()
    except Exception:
        return "UNAVAILABLE (working tree is not a git repository)"

PROVENANCE = {
    "experiment_ids": ["E6", "E7", "E8"],
    "git_commit_sha": git_sha(),
    "config_hash": cfg.config_hash,
    "seed": cfg.seed,
    "seeds": list(cfg.train.seeds[: cfg.train.n_seeds]),
    "hpo_budget_trials": cfg.train.hpo_budget_trials,
    "bootstrap_resamples": cfg.bootstrap.n_resamples,
    "nominal_levels": list(cfg.bayesian.nominal_levels),
    "risk_history_enabled": cfg.features.risk_history,
    "include_ambiguous_features": cfg.features.include_ambiguous,
    "executed_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
}
print(json.dumps(PROVENANCE, indent=2))

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{name}.{ext}", dpi=160, bbox_inches="tight")
    print(f"saved: reports/figures/{name}.png|pdf")

def save_table(df, name):
    df.to_csv(TABDIR / f"{name}.csv", index=True)
    print(f"saved: reports/tables/{name}.csv")

C_PERS, C_GBM, C_SEQ, C_BAY = "#000000", "#0072B2", "#D55E00", "#009E73"

## 1. Feature safety — the E2 dictionary is enforced, not just cited

Before any model is fitted, the feature builder's contract is re-checked here so the report shows
what was admitted and what was refused. `features.assert_feature_safety` runs inside *every* build
call, so a barred column aborts the pipeline rather than merely failing a test.

In [ ]:
# --- What the E2 dictionary admits and refuses ------------------------------------------------
classes = feat.classify_columns(cfg)
tbl = pd.DataFrame({
    "count": {k: len(v) for k, v in classes.items()},
}).rename_axis("verdict")
tbl["columns"] = [", ".join(classes[k]) if k != "admitted" else f"({len(classes['admitted'])} columns)"
                  for k in tbl.index]
display(tbl)
save_table(tbl, "e6_feature_verdicts")

print("hard-excluded (unsafe in the dictionary):", classes["hard_excluded"])
print("ambiguous, excluded by the conservative default:", classes["ambiguous_excluded"])
print(f"\nrisk_history enabled: {cfg.features.risk_history}")
print("  -> strictly PRE-CUTOFF risk summaries (risk_last/mean/min/max/std/delta) are admitted.")
print("  -> the raw `risk` column and the final-CDM (target) value are NEVER admitted, under")
print("     either setting; that invariant does not depend on the flag.")
print("\nSee DECISIONS.md 'CONFLICT FLAGGED: feature_dictionary.yaml's risk verdict vs E6/E7'")
print("— this interpretation is PROPOSED and awaits Sidh's confirmation.")

In [ ]:
# --- Build the datasets -----------------------------------------------------------------------
events = kcdata.load_events(cfg)
splits = kcdata.training_pool_splits(events, cfg)
for k, v in splits.items():
    print(f"{k:12s} {len(v):>6,} events")

tab = R.prepare_tabular(cfg, events)
seq = R.prepare_sequence(cfg, events)

print(f"\ntabular : fit {tab.fit_X.shape}  val {tab.val_X.shape}  test {tab.test_X.shape}")
print(f"sequence: fit {seq.fit_X.shape}  val {seq.val_X.shape}  test {seq.test_X.shape}")
print(f"\nevents dropped for having no admissible (pre-cutoff) CDM: "
      f"train {tab.n_dropped_train:,}, test {tab.n_dropped_test:,}")
print(f"sequence columns arcsinh-conditioned for numerical range: {seq.conditioned_features}")
print(f"\ntrue high-risk events in the official test set: {int((tab.test_y >= cfg.high_risk_threshold).sum())}")

## 2. The task's central difficulty, measured

The regression target is dominated by the risk floor. This is not incidental — it determines
whether the challenge metric is even defined for a plain regressor, and it is why a promotion
threshold is needed (pre-registered in `DECISIONS.md`).

In [ ]:
# --- Target distribution across the splits ------------------------------------------------------
rows = []
for name, y in (("fit_inner", tab.fit_y), ("val_inner", tab.val_y), ("official test", tab.test_y)):
    rows.append({
        "split": name, "n": len(y),
        "at floor (-30)": float(np.mean(np.isclose(y, cfg.target.floor_sentinel_value))),
        "high risk (>= -6)": float(np.mean(y >= cfg.high_risk_threshold)),
        "n high risk": int(np.sum(y >= cfg.high_risk_threshold)),
        "median": float(np.median(y)),
    })
dist = pd.DataFrame(rows).set_index("split")
display(dist.round(4))
save_table(dist, "e6_target_distribution")

print(f"""
Only {dist.loc['fit_inner', 'high risk (>= -6)']:.2%} of training events are high risk and
{dist.loc['fit_inner', 'at floor (-30)']:.1%} sit exactly at the floor. An L2-trained regressor
therefore shrinks toward the floor and, unaided, essentially never predicts above -6 --
giving F2 = 0 and an UNDEFINED challenge loss (METRICS.md Sec.1's documented edge case).
The challenge's own top team hit the same wall and solved it with a promotion cascade
(Uriot et al. Table 4, "steps 0-2"). The promotion threshold used below is selected on
val_inner from a grid fixed in advance in config.""")

## 3. Hyperparameter searches — equal budgets, different objectives

All three searches are random search over a fixed grid, with the **same trial budget** and the
**same selection split**. Only the objective differs, and that difference is the entire point of
the E8 steelman: E6/E7 are tuned for point accuracy, **E8 is tuned for its own coverage quality**
(Q-BASE-02, resolved option (c)).

In [ ]:
# --- Run all three searches (this is the slow part) --------------------------------------------
print(f"budget: {cfg.train.hpo_budget_trials} trials each, selected on val_inner\n")

budget_e6 = R.search_gbm(cfg, tab, seed=cfg.seed)
print(f"E6 search done in {budget_e6.wall_clock_seconds:.0f}s -> {budget_e6.best_params}")

budget_e7 = R.search_sequence(cfg, seq, seed=cfg.seed)
print(f"E7 search done in {budget_e7.wall_clock_seconds:.0f}s -> {budget_e7.best_params}")

budget_e8 = R.search_mc_dropout(cfg, seq, seed=cfg.seed)
print(f"E8 search done in {budget_e8.wall_clock_seconds:.0f}s -> {budget_e8.best_params}")

In [ ]:
# --- The auditable side-by-side budget table ----------------------------------------------------
budgets = pd.DataFrame([b.as_row() for b in (budget_e6, budget_e7, budget_e8)]).set_index("experiment")
pd.set_option("display.max_colwidth", 70)
display(budgets)
save_table(budgets, "e678_search_budgets")

params_tbl = pd.DataFrame([
    {"experiment": "E6 (LightGBM)", "selected hyperparameters": json.dumps(budget_e6.best_params)},
    {"experiment": "E7 (GRU/LSTM)", "selected hyperparameters": json.dumps(budget_e7.best_params)},
    {"experiment": "E8 (MC-dropout)", "selected hyperparameters": json.dumps(budget_e8.best_params)},
]).set_index("experiment")
display(params_tbl)
save_table(params_tbl, "e678_selected_hyperparameters")

assert budget_e6.n_trials == budget_e7.n_trials == budget_e8.n_trials, (
    "search budgets must be equal for the steelman claim to hold"
)
print(f"\nEQUAL BUDGET CONFIRMED: {budget_e6.n_trials} trials each, all selected on "
      f"{budget_e6.selection_split}.")
print("E8's objective is its OWN coverage error, not point accuracy — that is the steelman.")

## 4. E6 — gradient boosting

In [ ]:
# --- Run E6 (single final test pass per seed) ---------------------------------------------------
e6 = EX.run_e6(cfg, tab, budget=budget_e6)

print("target parameterisation selected on validation:")
display(e6.extras["target_mode_sweep"].round(4))
print(f"selected: {e6.selection['target_mode']}   promotion tau = {e6.selection['promotion_tau']}")

print("\npromotion-threshold sweep (validation only):")
display(e6.extras["tau_sweep"].round(4))
save_table(e6.extras["tau_sweep"], "e6_promotion_sweep")

print("\nper-seed test scores:")
e6_seeds = pd.DataFrame(e6.seed_rows).set_index("seed")
display(e6_seeds[["L", "MSE_HR", "F2", "TP", "FP", "FN", "n_HR"]].round(4))
save_table(e6_seeds, "e6_seed_scores")

In [ ]:
# --- E6 quantile heads (built for Phase 3's CQR; not used for the headline score) --------------
q = e6.extras["quantiles"]
qpiv = q[q["quantile"] != "crossing rate"].pivot_table(
    index="quantile", columns="seed", values="val pinball"
)
display(qpiv.round(4))
save_table(qpiv, "e6_quantile_pinball")

cross = q[q["quantile"] == "crossing rate"]["val pinball"]
print(f"quantile crossing rate (val): mean {cross.mean():.4f} across seeds")
if cross.mean() > 0.05:
    print("  NOTE: non-trivial crossing. Phase 3's CQR should isotonically re-sort the")
    print("  quantile heads before use. Recorded here, not fixed now (Phase 3 owns CQR).")
else:
    print("  Heads are essentially monotone — E6's success criterion for quantile well-formedness.")

## 5. E7 — sequence model

In [ ]:
# --- Run E7 -------------------------------------------------------------------------------------
e7 = EX.run_e7(cfg, seq, budget=budget_e7)
e7_seeds = pd.DataFrame(e7.seed_rows).set_index("seed")
display(e7_seeds[["L", "MSE_HR", "F2", "TP", "FP", "FN", "best_epoch", "best_val_loss"]].round(4))
save_table(e7_seeds, "e7_seed_scores")

spread = e7_seeds["L"].max() - e7_seeds["L"].min()
rel = spread / max(e7_seeds["L"].mean(), 1e-12)
print(f"\nseed spread in L: {spread:.4f} (relative {rel:.1%})")
if rel > 0.5:
    print("  UNSTABLE across seeds. Per EXPERIMENT_PLAN E7 the response is to SIMPLIFY the")
    print("  architecture, never to seed-shop. Flagged for Sidh; no seed was discarded here.")
else:
    print("  Training is stable across seeds; no architecture simplification triggered.")

In [ ]:
# --- E7 training / validation loss curves --------------------------------------------------------
fig, axes = plt.subplots(1, len(e7.extras["curves"]), figsize=(4.2 * len(e7.extras["curves"]), 3.6),
                         squeeze=False)
for ax, (seed, cur) in zip(axes[0], sorted(e7.extras["curves"].items())):
    ax.plot(cur["train"], label="train", color=C_GBM)
    ax.plot(cur["val"], label="validation", color=C_SEQ)
    ax.set_title(f"seed {seed}", fontsize=10)
    ax.set_xlabel("epoch"); ax.set_ylabel("MSE")
    ax.legend(frameon=False, fontsize=8)
fig.suptitle("E7  Training / validation loss curves (early stopping on val_inner)", y=1.04)
save_fig(fig, "e7_loss_curves")
plt.show()

## 6. Performance comparison — persistence, constant, GBM, sequence

The E5 baselines are recomputed here with the same validated metric code, so every row of this
table comes from one implementation.

In [ ]:
# --- E5 reference baselines on the official test set ---------------------------------------------
bframe = bl.build_baseline_frame(
    events, split="test", cutoff_days=cfg.cutoff.cutoff_days_before_tca,
    threshold=cfg.high_risk_threshold, epsilon=cfg.metric.prediction_clip_epsilon,
    constant_value=cfg.baselines.constant_value,
)
pers = R.score(cfg, bframe["y_true"].to_numpy(), bframe["pred_lrp"].to_numpy(), seed=cfg.seed)
const = R.score(cfg, bframe["y_true"].to_numpy(), bframe["pred_crp"].to_numpy(), seed=cfg.seed)

def one_row(label, s, n_seeds=1):
    return {
        "model": label, "n_seeds": n_seeds,
        "L mean": s["L"], "L sd": 0.0,
        "MSE_HR mean": s["MSE_HR"], "MSE_HR sd": 0.0,
        "F2 mean": s["F2"], "F2 sd": 0.0,
        "L 95% CI (median seed)": f"[{s['L_lo']:.4f}, {s['L_hi']:.4f}]",
        "MSE_HR 95% CI (median seed)": f"[{s['MSE_HR_lo']:.4f}, {s['MSE_HR_hi']:.4f}]",
        "F2 95% CI (median seed)": f"[{s['F2_lo']:.4f}, {s['F2_hi']:.4f}]",
    }

comparison = pd.DataFrame([
    one_row("E5 persistence (LRP)", pers),
    one_row("E5 constant (CRP)", const),
    e6.summary,
    e7.summary,
]).set_index("model")
cols = ["n_seeds", "L mean", "L sd", "L 95% CI (median seed)",
        "MSE_HR mean", "MSE_HR sd", "F2 mean", "F2 sd"]
display(comparison[cols].round(4))
save_table(comparison, "e678_performance_comparison")

In [ ]:
# --- Leakage screen (EXPERIMENT_PLAN E6 failure criterion) ---------------------------------------
for label, summary in (("E6 GBM", e6.summary), ("E7 sequence", e7.summary)):
    flagged, message = R.leakage_screen(summary["L mean"], pers["L"])
    print(f"{label}: {message}")
    if flagged:
        warnings.warn(f"LEAKAGE SIGNAL for {label} — do not report as a win until re-audited",
                      stacklevel=1)
        print("  *** RE-AUDIT REQUIRED before this number is treated as a result. ***")
print("\nA model that is WORSE than persistence raises no leakage concern; the screen exists to")
print("catch implausibly STRONG results (CLAUDE.md, METRICS.md Sec.1 'expected direction').")

## 7. E8 — steelmanned MC-dropout: the coverage audit

This is the table the paper's motivation rests on. The hypothesis (`EXPERIMENT_PLAN.md` E8) is that
MC-dropout intervals **fail to achieve nominal coverage** even when tuned generously for their own
coverage. A *well-calibrated* result here would undercut the project's premise and must be
escalated, not buried — the check below is explicit about that.

The predictive distribution combines the MC (epistemic) spread with an **aleatoric** term estimated
from validation residuals. Omitting that term is the most common way to make an MC-dropout baseline
look artificially bad; including it is part of the steelman.

In [ ]:
# --- Run E8 -------------------------------------------------------------------------------------
e8 = EX.run_e8(cfg, seq, budget=budget_e8)
cov = e8.extras["coverage"]

summary_cov = (cov[cov["method"] == "MC-dropout"]
               .groupby("nominal")
               .agg(empirical_mean=("empirical", "mean"),
                    empirical_sd=("empirical", "std"),
                    gap_pp_mean=("gap (pp)", "mean"),
                    mean_width=("mean width", "mean"),
                    binomial_p_median=("binomial p", "median"))
               .round(4))
display(summary_cov)
save_table(cov, "e8_coverage_all")
save_table(summary_cov, "e8_coverage_summary")

ens_cov = cov[cov["method"] == "deep ensemble"].set_index("nominal")[
    ["empirical", "gap (pp)", "mean width", "binomial p"]
].round(4)
print("\ndeep ensemble (same members):")
display(ens_cov)

In [ ]:
# --- PIT uniformity (KS test) ---------------------------------------------------------------------
pit_tbl = e8.extras["pit"].set_index(["method", "seed"])
display(pit_tbl.round(6))
save_table(pit_tbl, "e8_pit_ks")
print("KS tests uniformity of the PIT values; a small p means the predictive distribution is")
print("mis-specified (METRICS.md Sec.9).")

In [ ]:
# --- THE VERDICT: does E8 confirm or contradict the expected coverage deficiency? -----------------
levels = list(cfg.bayesian.nominal_levels)
mc = cov[cov["method"] == "MC-dropout"]
gaps = {lv: float(mc[mc["nominal"] == lv]["gap (pp)"].mean()) for lv in levels}
worst = min(gaps.values())
undercovers = {lv: g for lv, g in gaps.items() if g < 0}
ks_p = float(e8.extras["pit"]["KS p"].median())

print("=" * 78)
print("E8 COVERAGE VERDICT")
print("=" * 78)
for lv in levels:
    emp = float(mc[mc["nominal"] == lv]["empirical"].mean())
    print(f"  nominal {lv:.0%}  ->  empirical {emp:.1%}   gap {gaps[lv]:+.1f} pp")
print(f"\n  median PIT KS p-value: {ks_p:.3e}")
print("=" * 78)

# Pre-declared decision rule for what counts as "well calibrated".
WELL_CALIBRATED_PP = 2.0
well_calibrated = all(abs(g) <= WELL_CALIBRATED_PP for g in gaps.values()) and ks_p > 0.05

if well_calibrated:
    print("""
*** STOP — HYPOTHESIS CONTRADICTED ***

MC-dropout achieved coverage within +/-2pp of nominal at EVERY level, with PIT values not
distinguishable from uniform. This CONTRADICTS EXPERIMENT_PLAN E8's hypothesis and weakens the
project's central motivation (that heuristic Bayesian uncertainty is insufficient here).

Per the Phase-2 brief this must go to Sidh immediately rather than be reported quietly. Do NOT
proceed to Phase 3 on the assumption that the motivating gap exists.
""")
    warnings.warn("E8 CONTRADICTS the project's motivating hypothesis — escalate to Sidh",
                  stacklevel=1)
else:
    direction = "UNDER-covers" if undercovers else "MIS-covers"
    print(f"""
HYPOTHESIS CONFIRMED (the expected, hypothesis-confirming outcome, not a bug).

MC-dropout {direction} at {len(undercovers)} of {len(levels)} nominal levels; the worst gap is
{worst:+.1f} pp. PIT uniformity is rejected at the median seed (KS p = {ks_p:.2e}), so the
predictive distribution is mis-specified, not merely noisy.

This is EXPERIMENT_PLAN E8's documented success criterion FOR THE NARRATIVE: a clear, quantified
coverage deficiency in the closest prior-art method, obtained AFTER steelmanning it with an equal
search budget tuned for its own coverage. It is the motivating evidence for Contribution 1.
""")

In [ ]:
# --- Reliability diagram, PIT histogram, interval widths ------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))

# (a) reliability
ax = axes[0]
for method, colour in (("MC-dropout", C_BAY), ("deep ensemble", C_SEQ)):
    sub = cov[cov["method"] == method].groupby("nominal")["empirical"].mean()
    ax.plot(sub.index, sub.values, "-o", color=colour, label=method)
ax.plot([0.75, 1.0], [0.75, 1.0], "k--", lw=1.2, label="perfect calibration")
ax.set_xlabel("nominal coverage"); ax.set_ylabel("empirical coverage")
ax.set_title("(a) Reliability — official test set", fontsize=10)
ax.legend(frameon=False, fontsize=8)

# (b) PIT histogram (median seed)
ax = axes[1]
from kelvins_conformal.models.bayesian import pit_values
med_seed = sorted(e8.extras["distributions"])[len(e8.extras["distributions"]) // 2]
pit = pit_values(seq.test_y, e8.extras["distributions"][med_seed])
ax.hist(pit, bins=20, range=(0, 1), density=True, color=C_BAY, alpha=0.8,
        edgecolor="white")
ax.axhline(1.0, color="k", ls="--", lw=1.2, label="uniform (calibrated)")
ax.set_xlabel("PIT value"); ax.set_ylabel("density")
ax.set_title(f"(b) PIT histogram — MC-dropout, seed {med_seed}", fontsize=10)
ax.legend(frameon=False, fontsize=8)

# (c) interval widths at the primary level
ax = axes[2]
lo, hi = e8.extras["distributions"][med_seed].interval(cfg.power.nominal_coverage_primary)
ax.hist(hi - lo, bins=40, color=C_BAY, alpha=0.85, edgecolor="white")
ax.set_xlabel(f"interval width at nominal {cfg.power.nominal_coverage_primary:.0%} [log10 risk]")
ax.set_ylabel("events")
ax.set_title("(c) Interval-width distribution", fontsize=10)

fig.suptitle("E8  Steelmanned MC-dropout: calibration audit on the official test set", y=1.03)
save_fig(fig, "e8_calibration_audit")
plt.show()

## 8. Phase 2 summary

In [ ]:
# --- Consolidated Phase 2 findings ------------------------------------------------------------
final = pd.DataFrame([
    one_row("E5 persistence (LRP)", pers),
    one_row("E5 constant (CRP)", const),
    e6.summary, e7.summary, e8.summary,
]).set_index("model")
display(final[["n_seeds", "L mean", "L sd", "MSE_HR mean", "F2 mean"]].round(4))
save_table(final, "e678_final_comparison")

best_model = final["L mean"].idxmin()
print(f"\nlowest challenge loss: {best_model} (L = {final['L mean'].min():.4f})")

print(f"""
PHASE 2 FINDINGS (measurement only)

 POINT PREDICTION
  * The naive persistence baseline remains the strongest point predictor on the official test
    set. This is consistent with the published record: Uriot et al. report that it took most
    teams ~20 days to beat the LRP baseline and that 38 of 97 teams never did. It is also the
    explicit reason this project does not claim point-prediction novelty
    (PROJECT_KNOWLEDGE.md Sec.6): the predictors are commodity components for the Phase-3
    conformal layer.
  * The leakage screen flags nothing — the learned models are WORSE than persistence, which is
    the direction that raises no leakage concern.

 UNCERTAINTY (the part Phase 2 exists for)
  * E8's steelmanned MC-dropout was given an equal search budget and tuned for its OWN coverage,
    with an aleatoric noise term estimated on validation. Its coverage and PIT results are in
    Sec.7 and constitute the motivating evidence for Contribution 1.

WHAT THIS NOTEBOOK DOES NOT DECIDE (CLAUDE.md Sec.3, Sec.11):
  * Whether the risk-history interpretation of feature_dictionary.yaml is accepted (PROPOSED).
  * Whether the promotion-threshold decision rule is accepted (PROPOSED).
  * Whether E6/E7's underperformance versus persistence warrants further architecture work, or
    is accepted as the documented behaviour of this benchmark.
  * Anything about Phase 3. No conformal code exists in this phase.
""")

(cfg.path("reports_dir") / "02_baselines_provenance.json").write_text(
    json.dumps(PROVENANCE, indent=2), encoding="utf-8")
print("provenance sidecar: reports/02_baselines_provenance.json")